# The Hidden Layer: Solutions Notebook

This notebook contains the complete solution for `think_llm()`.

In [ ]:
# ── Setup (run this first) ────────────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/agentic_ai_spy")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")

In [ ]:
from hidden_layer import *
from hidden_layer.game_world import GameWorld, CellType, NPC, NPC_CATALOG
from hidden_layer.operative import Operative
from hidden_layer.agent import TOOLS_DESCRIPTION, parse_tool_call
from hidden_layer.oracle import ORACLE_TEMPLATE, llm_oracle
from hidden_layer.display import render_grid
print("The Hidden Layer loaded!")

In [ ]:
# Show the full map
operative, world, tools = create_game()
print(render_grid(world, operative, show_all=True))

---
## Play the Game Yourself!

Before looking at the AI solution, try playing the game manually to understand the base, the quests, and the challenge. Use the buttons to move, talk to informants, pick up items, and try to collect 10 dossiers!

In [ ]:
from hidden_layer.interactive import play_interactive

game = play_interactive()

---
## API Key Setup

In [ ]:
import os
from google import genai

# os.environ["GEMINI_API_KEY"] = "your-key-here"
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

---
## Solution: LLM Think Function

This solution uses a "middle ground" architecture:

- **BFS handles navigation** — LLMs are bad at spatial reasoning, so we never ask the LLM to choose cardinal directions. Instead, the LLM outputs a target coordinate like `NAVIGATE: (5, 3)` and BFS finds the shortest path.
- **Auto-collect is deterministic** — if there are items on the current cell, just pick them up. No LLM call needed for this.
- **Everything else is LLM-driven** — the LLM reads its journal (past conversations), inventory, and cell descriptions to decide what to say, who to visit, what to craft, and when to fight. No hardcoded quest logic, no keyword lists, no pre-programmed strategy.

The LLM genuinely learns the game by playing it: it talks to informants, interprets their cryptic responses, remembers what it learned, and plans accordingly.

In [ ]:
import re
from collections import deque

# ── BFS helpers (deterministic navigation) ───────────────────────────────────────
# These are the ONLY hardcoded parts: pathfinding algorithms.
# LLMs genuinely cannot do grid-based spatial reasoning reliably.

def _bfs_next_step(start, target, world):
    """BFS pathfinding that navigates around walls."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == target:
            return path[0] if path else None
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _bfs_nearest_unvisited(start, operative, world):
    """BFS to find direction toward the nearest unvisited passable cell."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) not in operative.visited and path:
            return path[0]
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _bfs_explore(operative, world):
    """Fallback: BFS toward nearest unvisited cell, or any passable direction."""
    row, col = operative.position
    d = _bfs_nearest_unvisited((row, col), operative, world)
    if d:
        return f'TOOL: move(direction="{d}")'
    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'
    return 'TOOL: move(direction="north")'


# ── Response parser ──────────────────────────────────────────────────────────
# The LLM can output either:
#   TOOL: action(args)     → execute directly
#   NAVIGATE: (row, col)   → BFS navigates there
#   EXPLORE                → BFS to nearest unvisited

def _parse_navigate(text):
    """Extract a NAVIGATE: (row, col) target from LLM output."""
    match = re.search(r'NAVIGATE:\s*\(?\s*(\d+)\s*,\s*(\d+)\s*\)?', text)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    return None


# ── System prompt ────────────────────────────────────────────────────────────
# Notice: NO hardcoded quest info, NO keyword lists, NO NPC-specific tips.
# The LLM must figure everything out from the game context.

SYSTEM_PROMPT = """You are an AI agent controlling a spy operative infiltrating a military base on Isla Perdida.

GOAL: Collect 10 classified dossiers and survive (health > 0). You have at most 100 turns.

AVAILABLE TOOLS:
{tools}

HOW TO RESPOND — pick exactly ONE of these three formats:

1. DIRECT ACTION (when you're on a cell and want to interact):
   TOOL: talk(message="your message here")
   TOOL: collect()
   TOOL: fabricate(item="item name")
   TOOL: hide()

2. NAVIGATE TO A LOCATION (when you want to go somewhere):
   NAVIGATE: (row, col)
   The navigation system will find the shortest path for you.

3. EXPLORE (when you don't know where to go):
   EXPLORE
   The navigation system will guide you to the nearest unvisited cell.

IMPORTANT RULES:
- NEVER output move() — navigation is automatic via NAVIGATE or EXPLORE.
- You can only interact with what's on your CURRENT cell. To interact with something elsewhere, NAVIGATE there first.
- Read your JOURNAL carefully — informants give you crucial hints about what to do, where to go, and what items are needed.
- If you've already talked to someone and have nothing new, don't talk to them again — NAVIGATE or EXPLORE instead.
- WARNING: Moving into a robot cell WITHOUT the correct weapon deals 1 damage and bounces you back. Make sure you have the right weapon first!
- Facilities can craft weapons — talk to them to learn what they offer.

THINK STEP BY STEP:
1. What cell am I on? Is there something to do here?
2. What's in my inventory? Does it suggest a delivery or crafting opportunity?
3. What have I learned from informants? (Check the journal.)
4. Where should I go next to make progress toward 10 dossiers?

First write your REASONING, then your action on a new line."""


# ── The think function ─────────────────────────────────────────────────────────

def think_llm(operative: Operative, world: GameWorld, history: list[dict]) -> str:
    """LLM-powered think function — middle ground approach.
    
    Deterministic: BFS navigation, auto-collect items.
    LLM-driven: all decisions about what to say, craft, fight, and where to go.
    """
    row, col = operative.position
    cell = world.get_cell(row, col)
    cell_desc = cell.description if cell.description else cell.cell_type.label
    turns_used = len([h for h in history if h["role"] == "action"])

    # ── Auto-collect: if there are items here, just grab them ──
    # This is the ONLY autopilot rule. No point wasting an LLM call on this.
    if cell.items:
        return 'TOOL: collect()'

    # ── Build context for the LLM ──
    system = SYSTEM_PROMPT.format(tools=TOOLS_DESCRIPTION)

    # Recent action/result pairs
    action_lines = []
    for i, h in enumerate(history[-20:]):
        if h["role"] == "action":
            offset = len(history) - 20 + i if len(history) > 20 else i
            result_text = ""
            if offset + 1 < len(history) and history[offset + 1]["role"] == "result":
                result_text = f" → {history[offset + 1]['content'][:150]}"
            action_lines.append(f"  {h['content'][:100]}{result_text}")
    recent_actions = "\n".join(action_lines[-10:]) if action_lines else "  (first turn)"

    # Full journal — this is where the LLM learns about quests
    journal_text = "\n".join(
        f"  - {entry}" for entry in operative.journal
    ) if operative.journal else "  (no conversations yet — talk to informants!)"

    # Visited cells
    visited_str = ", ".join(f"({r},{c})" for r, c in sorted(operative.visited))

    # What's adjacent
    adjacent_info = []
    for direction, adj_cell in world.get_adjacent(row, col).items():
        if adj_cell is None:
            adjacent_info.append(f"  {direction}: edge of island (impassable)")
        else:
            adjacent_info.append(f"  {direction}: {adj_cell.cell_type.label} {adj_cell.cell_type.emoji}")
    adjacent_text = "\n".join(adjacent_info)

    user_msg = f"""== OPERATIVE STATUS ==
Position: ({row}, {col}) | Health: {operative.health}/3 | Dossiers: {operative.dossiers}/10
Turns used: {turns_used}/100 ({100 - turns_used} remaining)
Inventory: {operative.inventory if operative.inventory else '(empty)'}

== CURRENT CELL ==
{cell_desc}
Type: {cell.cell_type.label}

== ADJACENT CELLS ==
{adjacent_text}

== VISITED CELLS ({len(operative.visited)}/64) ==
{visited_str}

== RECENT ACTIONS ==
{recent_actions}

== JOURNAL (everything informants have told you) ==
{journal_text}

What do you do? Remember: reason first, then output exactly one TOOL:, NAVIGATE:, or EXPLORE line."""

    # ── Call the LLM ──
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=400,
                temperature=0.3,
            ),
        )
        text = response.text
    except Exception:
        return _bfs_explore(operative, world)

    # ── Parse the response ──

    # Check for NAVIGATE: (row, col)
    nav_target = _parse_navigate(text)
    if nav_target:
        d = _bfs_next_step((row, col), nav_target, world)
        if d:
            return f'TOOL: move(direction="{d}")'
        # Target unreachable or already there — explore instead
        return _bfs_explore(operative, world)

    # Check for EXPLORE
    if "EXPLORE" in text and "TOOL:" not in text:
        return _bfs_explore(operative, world)

    # Check for a TOOL: call
    tool_name, args = parse_tool_call(text)

    # Redirect move to BFS (LLM shouldn't be choosing directions)
    if tool_name == "move":
        return _bfs_explore(operative, world)

    return text

In [ ]:
oracle_fn = lambda npc, q, o: llm_oracle(npc, q, o, client)

result = play_with_llm(
    think_fn=lambda operative, world, history: think_llm(operative, world, history),
    oracle_fn=oracle_fn,
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'} | Dossiers: {result['dossiers']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

In [ ]:
# Download the game log for debugging
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")